# Part i: Rebuilding the order book

**Data:** Intel (INTC), 21 June 2012, from LOBSTER

LOBSTER gives two files:
- the **message file**: one row for each event (a new order, a cancellation or a trade)
- the **orderbook file**: the book after each event.

Row k of one file matches row k of the other, so they must have the same number of rows.

In [ ]:
import pandas as pd

from unito26.lob.messages import BUY, SELL, limit_order, withdrawal
from unito26.lob.orderbook import AggregateBook

msg=pd.read_csv("../data/lobster/INTC_2012-06-21_34200000_57600000_message_10.csv", header=None, names=["Time","Type","OrderID","Size","Price","Direction"])

ob=pd.read_csv("../data/lobster/INTC_2012-06-21_34200000_57600000_orderbook_10.csv", header=None)

print(len(msg),len(ob))

display(msg.head())
display(ob.head())

## Three hours of trading

I use 10:00 to 13:00. I skip the first half hour after the open 09:30, because the book is still filling up.

Time is in seconds after midnight: 10:00 is 36 000 and 13:00 is 46 800. I keep the same rows in both files, so they still match row by row.

In [ ]:
#True for the rows between 10:00 and 13:00
keep=(msg["Time"]>=36000) & (msg["Time"]<=46800)

msg=msg[keep].reset_index(drop=True)
ob=ob[keep].reset_index(drop=True)

# How many rows are left and how many events of each type
print(len(msg),len(ob))
print(msg["Type"].value_counts())

What we have after cutting:

From 10:00 to 13:00 there are **274 356 events**. Both files still have the same number of rows. 

| Type | Event | Count | Share |
|---|---|---|---|
| 1 | new limit order | 135 002 | 49 % |
| 3 | deletion of an order | 124 093 | 45 % |
| 4 | trade with a visible order | 11 131 | 4 % |
| 2 | part of an order cancelled | 3 077 | 1 % |
| 5 | trade with a hidden order | 1 053 | 0.4 % |

Almost half of all events are new orders. Only about 4% of the events are trades.

## A book from one row of the orderbook file

Each row has 10 levels, and each level has 4 numbers: ask price, ask size, bid price and bid size

LOBSTER prices are dollars * 10 000. The package uses ticks (cents), so I divide the prices by 100.

In [ ]:
def book_at(k):
    row=ob.iloc[k]
    bids={}
    asks={}
    for i in range(10):
        ask_price=row[4*i]//100
        ask_size=row[4*i+1]
        bid_price=row[4*i+2]//100
        bid_size=row[4*i+3]
        asks[int(ask_price)]=int(ask_size)
        bids[int(bid_price)]=int(bid_size)
    return AggregateBook.from_levels(bids,asks)

# Check that I read row 0 correctly: both lines come from the file
book = book_at(0)
row = ob.iloc[0]

print("Our book:  bid", book.best_bid_price * 100, book.best_bid_size,
      "  ask", book.best_ask_price * 100, book.best_ask_size)
print("File:      bid", row[2], row[3], "  ask", row[0], row[1])

Both lines are the same: best bid 27.34 dollars with 3 437 shares, best ask 27.35 dollars with 4 470 shares. So 'book_at' reads the colums and the prices correctly and I can use it as the starting book and as the answer key later.

## From a LOBSTER row to a message

Each row of the message file becomes one message for the book:

| Type | What happened | Message |
|---|---|---|
| 1 | new order | `limit_order`: add shares at the price |
| 2, 3 | order (partly) cancelled | `withdrawal`: remove shares at the price |
| 4 | trade | `withdrawal`: remove shares from the order that was hit |
| 5, 7 | hidden trade, trading halt | `withdrawal` of 0 shares: nothing changes |

For a trade, 'Direction' is the side of the order that was hit, so that is the side I remove the shares from.

I still make a message for types 5 and 7, so that the rows keep matching the file.

In [ ]:
def to_message(k):
    m=msg.iloc[k]
    if (m["Direction"] == 1):
        side = BUY
    else:
        side = SELL

    size=int(m["Size"])
    price=int(m["Price"])//100

    if m["Type"]==1:
        return limit_order((m["Time"]), size, price, side)
    if m["Type"] in (2,3,4):
        return withdrawal((m["Time"]), size, price, side)
    return withdrawal(m["Time"], 0, 0, side)

for k in range (5):
    print(to_message(k))

The first five messages look right:
- rows 0,2 and 4 are new buy orders ('SUBMIT') of 100, 1500 and 200 shares
- row 1 removes 1400 shares from the sell side at 27.35 dollars ('WITHDRAW')
- row  3 removes 100 shares from the buy side at 27.32 dollars ('WITHDRAW')

The prices are in ticks: 2734 means 27.34 dollars

## TEST 1: one message at a time

I check that each message changes the book in the right way. For message k I start from the book in row k-1 of the file, apply message k and compare with row k of the file.

I compare the top of the book: the best bid, the best ask, and the sizes there. 

In [ ]:
def top(book):
    return (book.best_bid_price, book.best_bid_size, book.best_ask_price, book.best_ask_size)

# Check messages 1 to 5, one at a time
for k in range (1,6):
    book=book_at(k-1)
    book.apply(to_message(k), record=False)

    # My top of the book, the file's top of the book and wheteher they are the same
    print(k, top(book), top(book_at(k)), top(book) == top(book_at(k)))

All five messages match.

For example, message 1 removes 1400 shares at the best ask (4470-1400 = 3070)

Now I can do the same check for every message from 10:00 to 13:00, and count how many messages give a different top of the book than the file

In [ ]:
wrong= []

for k in range (1, len(msg)):
    book=book_at(k-1)
    book.apply(to_message(k), record=False)
    if top(book) != top(book_at(k)):
        wrong.append(k)

print("Messages checked:", len(msg)-1)
print("Messages with a different top of the book:", len(wrong))

## TEST 2: one book through all three hours

I take row 0 from the file only once. After that, only the messages change the book.

In [ ]:
book = book_at (0)
wrong = []

for k in range (1, len(msg)):
    book.apply(to_message(k), record=False)
    if top(book) != top(book_at(k)):
        wrong.append(k)

print("Rows checked:", len(msg) - 1)
print("Rows with a different top of the book:", len(wrong))
print("Share:", round(len(wrong) / (len(msg) - 1), 3))
print("First different row:", wrong[0] if wrong else None)
print("At the end, ours:", top(book))
print("At the end, file:", top(book_at(len(msg) - 1)))

The top of the book differs from the file in 40% of the rows, but it matches again at 13:00. Test 1 found no wrong message, so the book must be missing something the file knows.

## The first difference

I look at row 12 698, the first row where the top is different.

In [ ]:
first = wrong[0] # The first row where the top is different 

book = book_at(0)
for k in range (1, first + 1):
    book.apply(to_message(k), record=False)

file_book = book_at(first)

display(msg.iloc[first-3 : first + 1])

print("Asks, ours:", sorted(book.asks.items())[:3])
print("Asks, file:", sorted(file_book.asks.items())[:3])
print("Bids, ours:", sorted(book.bids.items(), reverse=True)[:3])
print("Bids, file:", sorted(file_book.bids.items(), reverse=True)[:3])

## Why the book is different

At row 0 (10:00) the 10 bid levels go down to 27.25 dollars. The shares already resting at 27.24 dollars and below were not in the file, so the book never knew about them.

At row 12 698 a large sell order empties 27.25 and 27.24 becomes the best bod: the price is right, but the book has 500 shares too few there.

If this is the only reason, the wrong rows should be the rows where the price is outside what I knew at 10:00.

In [ ]:
start = book_at(0)
lowest_bid = min(start.bids)
highest_ask = max(start.asks)
print("Known at 10:00: bids down to", lowest_bid, " asks up to", highest_ask)

best_bid = ob[2] // 100
best_ask = ob[0] // 100
outside = (best_bid < lowest_bid) | (best_ask > highest_ask)

is_wrong = pd.Series(False, index=ob.index)
is_wrong[wrong] = True

print(pd.crosstab(outside, is_wrong,
                  rownames=["Outside the prices known at 10:00"],
                  colnames=["The top is different"]))

Only 72 091 of the 110 904 different rows (65%) are outside the known prices. But 38 813 rows are inside, so there must be a second reason. I look at the first such row.

In [ ]:
# The rows where the top is different, but the price is inside what were known at 10:00
inside_wrong = [k for k in wrong if not outside[k]]
first = inside_wrong[0]
print("First different row inside the known prices:", first)

# Build our book again up to that row
book = book_at(0)
for k in range(1, first + 1):
    book.apply(to_message(k), record=False)

file_book = book_at(first)

# The last four messages, and the best three levels on each side
display(msg.iloc[first - 3 : first + 1])
print("Asks, mine:", sorted(book.asks.items())[:3])
print("Asks, file:", sorted(file_book.asks.items())[:3])
print("Bids, mine:", sorted(book.bids.items(), reverse=True)[:3])
print("Bids, file:", sorted(file_book.bids.items(), reverse=True)[:3])

## A second reason

At row 30 987 my book has too many shares on the ask side: 2300 more at 27.32, 1800 more at 27.33 and 3600 more at 27.34. 

The first reason (orders I never saw) can only give too few shares. So here the file must have lost shares without any messages saying so.

I check when the difference at 27.32 happened, and if the file showed that price just before.

In [ ]:
price = 2732             
book = book_at(0)
last_seen = None          # the last row where the file showed this price

for k in range(1, first + 1):
    book.apply(to_message(k), record=False)
    file_size = book_at(k).asks.get(price)       # None if the price is not among the 10 levels
    ours = book.asks.get(price, 0)

    if file_size is None:                        # the file does not show this price: skip
        continue
    if file_size != ours:                       
        print("First row where", price, "differs:", k, "| ours", ours, "| file", file_size)
        print("Last row before that where the file showed", price, ":", last_seen)
        break
    last_seen = k                                # the file showed the price, agreed

The files showed 27.32 at row 18 149, and the book agreed. Between rows 18 150 and 18 754 the price was not among the 10 levels of the file. When it came back at row 18 755, the file had 1900 fewer shares than the book (4700 against 6600)

So the difference appeared while the price could not be seen in the file. In that time the file lost shares that no message removed. I do not know why LOBSTER does this.

## Summary of part i

I rebuilt the order book of INTC from the message file, from 10:00 to 13:00 and compared it with the orderbook file.

**TEST 1**: all 274 355 messages are read correctly

**TEST 2**: my book is different from the file in 40% of the rows. I found two possible reasons:

1. Orders resting before 10:00 that I never saw (too few shares)
2. Changes in the file without a message (too many shares)

Once a price is wrong, it stays wrong there, aso the error shows up again and again. This is possibly why so many rows are different.

# Part ii: The queue imbalance and the order flow imbalance

Both use only the top of the book: the best bid and ask prices, and the size resting there. I take it from the orderbook file, since part i showed that my book differs in 40% of the rows. Each row of the file is one event.

In [ ]:
import numpy as np

top_df = pd.DataFrame({
    "Time":msg["Time"],         
    "BidPrice":ob[2]//100,      # best bid price
    "BidSize":ob[3],            # shares at the best bid
    "AskPrice":ob[0]//100,      # best ask price
    "AskSize":ob[1]})            # shares at the best ask

# Mid-price
top_df["Mid"] = (top_df["BidPrice"] + top_df["AskPrice"]) / 2

display(top_df.head())

The first rows show the book at 10:00: 
- best bid 27.34
- best ask 27.35
- mid-price 27.345

## The queue imbalance I¹

The queue imbalance at the best level, with n = 1:

$$I^1_t = \frac{S^{b,1}_t - S^{a,1}_t}{S^{b,1}_t + S^{a,1}_t}$$

It is +1 when there are only buyers at the top, -1 when there are ontly sellers and 0 when the two queues are equal.


In [ ]:
top_df["I1"] = ((top_df["BidSize"] - top_df["AskSize"]) / (top_df["BidSize"] + top_df["AskSize"]))

display(top_df[["BidSize", "AskSize", "I1"]].head())

At 10:00 the sell queue is larger, so I¹ is negative (-0.13). After 1400 shares leave the ask and 1500 join the bid, I¹ turns positive (0.23)

## The contribution e_n of each event

Compares the top of the book after event n with the top just before it:

$$e_n = 1_{\{P^b_n \ge P^b_{n-1}\}} S^{b}_n - 1_{\{P^b_n \le P^b_{n-1}\}} S^{b}_{n-1}
      - 1_{\{P^a_n \le P^a_{n-1}\}} S^{a}_n + 1_{\{P^a_n \ge P^a_{n-1}\}} S^{a}_{n-1}$$

On the bid side there are three cases:
- the best bid does not move
- the best bid goes up
- the best bid goes down

Positive e_n means buying pressure and negative means selling pressure

In [ ]:
prev = top_df.shift(1)

bid_part = ((top_df["BidPrice"] >= prev["BidPrice"]) * top_df["BidSize"]
            - (top_df["BidPrice"] <= prev["BidPrice"]) * prev["BidSize"])

ask_part = ((top_df["AskPrice"] <= prev["AskPrice"]) * top_df["AskSize"]
            - (top_df["AskPrice"] >= prev["AskPrice"]) * prev["AskSize"])

top_df["e"] = bid_part - ask_part
top_df.loc[0, "e"] = np.nan             # the first row has no row before it

display(top_df[["BidPrice", "BidSize", "AskPrice", "AskSize", "e"]].head())

- Row 1: the best bid does not move (0), and 1400 ahres leave the best ask. Fewer sellers counts as buying pressure: e=+1400
- Row 2: 1500 shares join the best bid: e=+1500
- Rows 3 and 4: the top of the book does not change, so e=0

## The order flow imbalance OFI

OFI over a window of w seconds is the sum of e_n over the events in (t-w, t]:

$$\mathrm{OFI}_{t,w} = \sum_{n:\ t-w < T_n \le t} e_n$$

I use two functions from sample question 3.6: 'window_start' finds the first event of each window, and 'rolling_sum' adds up the values inside it. A window that reaches back to the first row would also need events from before  10:00, which I do not have, so I leave it empty (NaN).

In [ ]:
def window_start (times, window):
    return np.searchsorted(times, times - window, side="right")

def rolling_sum (start, values):
    cumulative = np.concatenate(([0.0], np.cumsum(values)))
    return cumulative[1:] - cumulative[start]

times = top_df["Time"].to_numpy()
e = top_df["e"].fillna(0).to_numpy()

for w in (10, 60, 300):
    start = window_start(times, w)
    ofi = rolling_sum(start, e)
    ofi[start == 0] = np.nan
    top_df[f"OFI{w}"] = ofi

display(top_df[["Time", "e", "OFI10", "OFI60", "OFI300"]].iloc[50000:50005])

At 10:26:40 all three OFI are negative: over the last 10 secods, 1 minute and 5 minutes there was more selling than buyins pressure at the top of the book.

When an event adds e=+200 (row 50 004), all three OFI goes up by exactly 200, because OFI is the sum of e over the window.

## Checking OFI by hand

I check one value by adding up e directly over the window, without the two functions. 

In [ ]:
k = 50004               # row I check
w = 60                  # the window

inside = (top_df["Time"] > top_df["Time"][k] - w) & (top_df.index <= k)

print("By hand:", top_df["e"][inside].sum())
print("rolling_sum:", top_df["OFI60"][k])

## What I¹ and OFI look like from 10:00 to 13:00

In [ ]:
display(top_df[["I1", "OFI10", "OFI60", "OFI300"]].describe().round(2))

print("Mid at 10:00:", top_df["Mid"].iloc[0], "  Mid at 13:00:", top_df["Mid"].iloc[-1])

- I¹ lies between −1 and 1. The mean (0.04) and median (0.05) are close to 0, which means that on average the buy and sell queues at the top are about the same size.
- The mean of OFI is negative, and more negative for longer windows. Indicates that from 10:00 to 13:00 there was more selling than buying pressure at the top of the book.
- The mid-price fell from $27.345 to $26.935. This goes well with the selling pressure,
- The std of OFI grows with the window, most likely because a longer window adds up more events.

# Part iii: Mid-price change against OFI

Cut 10:01 - 13:00 into windows of 60 seconds that do not overlap and take the change in the mid_price and the OFI in each.

In [ ]:
w = 60
grid = np.arange (36060, 46801, w)                      #the timestamps in seconds
idx = np.searchsorted(times, grid, side="right") - 1    #the last row at or before each time

mid = top_df["Mid"].to_numpy()
cs = np.cumsum(e)           

#change in the mid-price and sum of e inside the window
win = pd.DataFrame({"dMid": mid[idx[1:]] - mid[idx[:-1]], "OFI": cs[idx[1:]] - cs[idx[:-1]]})

print(len(win),"windows")
display(win.head())

OFI and ΔP^m both show selling pressure and the mid-rpice goes down. The more negative OFI is, the more mid-price falls.

## Regression of ΔP^m on OFI

In [ ]:
import matplotlib.pyplot as plt

# dMid = intercept + slope * OFI
slope, intercept = np.polyfit(win["OFI"], win["dMid"], 1)

# the share of the variation in dMid that OFI explains
r2 = np.corrcoef(win["OFI"], win["dMid"])[0, 1]**2

print("slope:", slope, "intercept:", round(intercept, 3), "R²:", round(r2, 3))

plt.scatter(win["OFI"], win["dMid"], s=10)
plt.plot(win["OFI"], intercept + slope * win["OFI"], color="red")
plt.xlabel("OFI (shares)")
plt.ylabel("ΔP^m (ticks)")
plt.show()

OFI explains 89% of the mid-rpice changes. 

## The depth implied by the slope

In (1.4.3) the slope is τ/(2S̄), with τ = 1 tick. So the slope implies a depth

S̄ = 1 / (2 · slope)

I compare it with the sizes at the top of the book in the file.

In [ ]:
implied_depth = 1 / (2*slope)                   # S̄ from slope = τ / (2 S̄)
print("Depth implied by the slope:", round(implied_depth))

print("Average best bid size:", round(top_df["BidSize"].mean()))
print("Average best ask size:", round(top_df["AskSize"].mean()))

The slope implies around S̄ ≈ 20 900 shares. This is a lot more than the average size at the top of the book. Goes well with the assumption that queus behind the top hold S̄, while the queue at the top holds at most S̄.

## The residual term


In [ ]:
sb = top_df["BidSize"].to_numpy()
sa = top_df["AskSize"].to_numpy()

# Change in the size at the best bid and best ask over each window
win["dSb"] = sb[idx[1:]] - sb[idx[:-1]]
win["dSa"] = sa[idx[1:]] - sa[idx[:-1]]


win["X"] = win["OFI"] - win["dSb"] + win["dSa"]

slope2, intercept2 = np.polyfit(win["X"], win["dMid"], 1)
r2_2 = np.corrcoef(win["X"], win["dMid"])[0, 1] ** 2

print("OFI alone: slope", slope, "R²", round(r2, 3), "depth", round(1 / (2 * slope)))
print("OFI with (1.4.4): slope", slope2, "R²", round(r2_2, 3), "depth", round(1 / (2 * slope2)))

The adjusted R-square goes from 0.888 to 0.919, and the implied depth is 19 400. OFI leaves 11.2 % of the mid-price changes unexplained.

# Part iv: Does OFI forecast the mid-price? 

In [ ]:
def row_at(t):
    return np.searchsorted(times, t, side="right") - 1

def forecast(w, h):
    step = max(w,h)                              # spacing so windows do not overlap
    t = np.arange(36001+w, 46800-h+1, step)    # times t, with room for w before and h after
    ofi = cs[row_at(t)]-cs[row_at(t-w)]
    future = mid[row_at(t+h)] - mid[row_at(t)]
    corr = np.corrcoef(ofi, future)[0,1]        # the sign and strenght of the correlation
    return corr, len(t)

print(forecast(60, 60))

For w = h = 60 seconds the correlation is -0.04, far below the rule of thumb limit 2/√178 ≈ 0.15. 

## A grid of windows and horizons

I compute the correlation for w and h in 10, 30, 60 and 300 seconds.

In [ ]:
rows = []
for w in (10, 30, 60, 300):
    for h in (10, 30, 60, 300):
        corr, n = forecast(w,h)
        readable = abs(corr)>2/np.sqrt(n)                   # is it different from 0?
        rows.append({"w":w, "h":h, "corr": round(corr,3), "n":n, "readable": readable})

grid_df = pd.DataFrame(rows)

display(grid_df.pivot(index="w", columns="h", values="corr"))

display(grid_df.pivot(index="w", columns="h", values="readable"))

## Summary of part iv

- The correlations between OFI and the mid-price change lie between -0,27 and 0,17. None can be told apart from 0.
- Slightly positive for short windows, so the price may cintinue a little in the direction of OFI.
- So over the three hours OFI does not forecast the mid-price at any point.

# Part v: Does I¹ forecast the mid-price

I do the same in part iv with the queue imalance I¹ at time t. I¹ have no wnindow, so I only vary h.

But in generated data any forecast from I¹ is only mechanical, so OFI and I¹ cannot be compared fairly. Here the data is real trading: a thin queue may be thin because traders know something.
So the comparison is fair in both directions.

In [ ]:
i1 = top_df["I1"].to_numpy()

rows = []
for h in (10, 30, 60, 300):                      
    t = np.arange(36001, 46800-h+1, h)
    imbalance = i1[row_at(t)]    
    future = mid[row_at(t+h)] - mid[row_at(t)]
    corr = np.corrcoef(imbalance, future)[0,1]    
    n = len(t)
    rows.append({"h":h, "corr I1": round(corr,3), "n":n, "readable": abs(corr)>2/np.sqrt(n)})

i1_df = pd.DataFrame(rows)

i1_df["corr OFI (w = h)"] = [grid_df[(grid_df.w == h) & (grid_df.h == h)]["corr"].iloc[0] for h in i1_df["h"]]

display(i1_df)

## Summary of part v

- The correlation between I¹ and the mid-price is positive at every horizon
- When there are more buyers than sellers at the top, the price tends to rise afterwards. The relation gets weaker for longer horizons.
- I¹ forecasts better than OFI at every horizon up to 60 s; OFI could not be read at any. 

Part of it may still be mechanical: a thin queue is one trade away from emptying, and these data cannot separate the two.